# Noble AI — Ejecutar y disfrutar en Google Colab

**Una sola celda:** al ejecutarla se instala el proyecto completo, se compila la interfaz web, se inicia FastAPI y se prepara un narrador local Ollama. Si Colab tiene GPU, el modo `auto` intenta instalar ComfyUI y un checkpoint inicial para ilustraciones.

El notebook abre Noble AI por el **proxy autenticado de esta sesión de Colab**. No crea un túnel público ni exige copiar comandos. Puedes elegir `Runtime → Run all` o ejecutar solo la siguiente celda.

**Configuración predeterminada:** `REVISION=master` descarga siempre la versión actual; `MODELO=auto` selecciona un modelo según el hardware; `IMAGENES=auto` intenta preparar imágenes solo si hay GPU; `COPIA_EN_DRIVE=False` no pide permisos. El primer arranque descarga dependencias y pesos; en reinicios se aprovechan los archivos disponibles.

> La sesión de Colab es temporal. Si necesitas conservar historias entre sesiones, activa `COPIA_EN_DRIVE` antes de ejecutar: solicitará acceso a Google Drive. Una copia periódica no equivale a sincronización instantánea ni evita todas las pérdidas por interrupciones repentinas.


In [ ]:
# @title ▶️ Instalar y abrir Noble AI — Ejecuta esta única celda
REVISION = "master" # @param {type:"string"}
MODELO = "auto" # @param {type:"string"}
IMAGENES = "auto" # @param ["auto", "on", "off"]
COPIA_EN_DRIVE = False # @param {type:"boolean"}

import argparse
import importlib.util
from pathlib import Path
import re
import subprocess
import sys
from urllib.request import urlopen

if REVISION != "master" and not re.fullmatch(r"[a-f0-9]{40}", REVISION):
    raise ValueError("REVISION debe ser master o un commit SHA completo de 40 caracteres")

# Resolver master una sola vez mantiene instalador y proyecto en el MISMO commit.
PINNED_SHA = REVISION
if REVISION == "master":
    row = subprocess.check_output(
        ["git", "ls-remote", "https://github.com/Mairon31/novel_ai.git",
         "refs/heads/master"], text=True, timeout=50).strip()
    PINNED_SHA = row.split()[0]
    if not re.fullmatch(r"[a-f0-9]{40}", PINNED_SHA):
        raise RuntimeError("No se pudo resolver la versión de master")
print("Noble AI - revisión fijada para este arranque:", PINNED_SHA[:12])
URL = ("https://raw.githubusercontent.com/Mairon31/novel_ai/"
       + PINNED_SHA + "/tools/colab_launch.py")
BOOTSTRAP = Path("/content/noble_ai_colab_launcher.py")
with urlopen(URL, timeout=60) as response:
    data = response.read(150_001)
if not data or len(data) > 150_000:
    raise RuntimeError("Instalador no disponible o de tamaño inesperado")
BOOTSTRAP.write_bytes(data)

spec = importlib.util.spec_from_file_location("noble_colab_launcher", BOOTSTRAP)
launcher = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = launcher
spec.loader.exec_module(launcher)
state = launcher.launch(argparse.Namespace(
    revision=PINNED_SHA, model=MODELO, images=IMAGENES,
    drive_backup=COPIA_EN_DRIVE,
))
print("\nNoble AI preparado. Commit:", state["source_revision"][:12])


## Ya está. ¿Cómo empiezo?

Al terminar la celda aparecerán **Abrir Noble AI en una pestaña** y una versión incrustada del lector. Usa Biblioteca para crear tu primera historia; el narrador Ollama habrá sido seleccionado automáticamente si pudo descargarse. El checkpoint de demostración usa Stable Diffusion 1.5 FP16: sirve para comprobar ilustraciones, pero no garantiza el estilo o calidad de modelos especializados.

Para actualizar Noble AI más adelante, vuelve a abrir este notebook con `REVISION = master`. Para repetir una versión específica, escribe su SHA completo. El instalador conserva las preferencias de narración/imágenes ya configuradas cuando continúan siendo válidas.

### Diagnóstico

Los estados se muestran durante la instalación y se registran en `/content/noble-ai-colab/logs/` (`python-deps.log`, `npm-ci.log`, `vite-build.log`, `noble-api.log`, `ollama*.log`, `comfyui.log`). Si la imagen opcional falla, el lector puede funcionar igualmente con el narrador. Revisa [la guía Colab](https://github.com/Mairon31/novel_ai/blob/master/docs/COLAB_PLAY.md).

No es necesario mantener una celda en bucle; los servidores quedan iniciados en el runtime. **Cuando Colab desconecte el runtime, los servidores se apagarán**. Si no se puede abrir la aplicación en una pestaña por políticas de cookies del navegador, utiliza el marco incrustado.
